In [1]:
%run T_symb.ipynb
%run helpers.ipynb

In [3]:
from sympy import IndexedBase
K=IndexedBase('K')


In [8]:
class Distr_of_constant_symbol(object):
    def __init__(self,symb,curv):
        """INPUTS:
        * 'symb' - a T_symb object
        * 'curv' - a positive 2-cochain from the cochain complex associated with symb"""
        
        self.Tanaka_symbol=symb
        self.curv=curv
        self.Jacobi_id_cache={}
        self.basis=[Vector_Field(A.vec,self) for A in symb.basis if A.wght<0]

    def bracket(self,v1,v2):
        """Returns [v1,v2] as a section of self
        INPUTS:
        * 'v1','v2' - Vector_Field objects with self as parent
        """
        g1,g2=[self.Tanaka_symbol.elt(Matrix(v1.vec)),self.Tanaka_symbol.elt(Matrix(v2.vec))]
        r=(g1.ad(g2)).vec
        e1e2=g1.cast_as_ext_elt().wedge(g2.cast_as_ext_elt())
        r+=self.curv.apply_cochain_map(e1e2).vec
        for i in range(len(v1.vec)):
            for j in range(len(v2.vec)):
                r[i]+=(v1.vec[j]*self.abn_ind_der(v2.vec[i],j)-v2.vec[j]*
                       self.abn_ind_der(v1.vec[i],j))
        return Vector_Field(r,self)
    
    def Jacobi_id(self,i,j,k,l):
        """Returns the Jacobi indentity in curvatures from cyc_{i,j,k}([[X_i,X_j],X_k]^l)
        """
        if (i,j,k,l) in self.Jacobi_id_cache: return self.Jacobi_id_cache[(i,j,k,l)]
        if (k,i,j,l) in self.Jacobi_id_cache: return self.Jacobi_id_cache[(k,i,j,l)]
        if (j,k,i,l) in self.Jacobi_id_cache: return self.Jacobi_id_cache[(j,k,i,l)]

        vi=Vector_Field([0]*i+[1]+[0]*(len(self.Tanaka_symbol.basis)-i-1),self)
        vj=Vector_Field([0]*j+[1]+[0]*(len(self.Tanaka_symbol.basis)-j-1),self)
        vk=Vector_Field([0]*k+[1]+[0]*(len(self.Tanaka_symbol.basis)-k-1),self)

        r=Vector_Field([0]*len(self.Tanaka_symbol.basis),self)
        ind_list=[vi,vj,vk,vi,vj,vk]

        for n in range(3): # cyclic sum
            a,b,c=ind_list[n:n+3]
            r+=self.bracket(self.bracket(a,b),c)
        i_list=[i,j,k]
        i_list.sort()
        for l1 in range(len(self.Tanaka_symbol.basis)):
            self.Jacobi_id_cache[tuple(i_list+[l1])]=self.normalize_der(r.vec[l1])
        return self.normalize_der(r.vec[l])


    def abn_ind_der(self,ind_expr,i):
        """Returns the derivative of ind_expr in the direction X_i among X_3, X_4,..., X_{2n-1},
        which is a frame on the base manifold.

        Note: This only works for horizontal derivatives!

        INPUTS:
        * 'ind_expr' -- an expression in h, e, y, and indexed objects
        * 'i' -- an integer between 3 and 2n-1
        """
        if type(ind_expr) in [Matrix, ImmutableDenseMatrix, MutableSparseMatrix, ImmutableSparseMatrix]:
            if type(ind_expr)==ImmutableDenseMatrix: result = mut_mat_copy(ind_expr)
            else: result = copy.copy(ind_expr)
            for j in range(shape(result)[0]):
                for k in range(shape(result)[1]):
                    result[j,k]=self.abn_ind_der(result[j,k],i)
            return result

        # # I'm not sure if this simplification will help or hurt time efficiency
        # ind_expr=simplify(ind_expr)
        if isinstance(ind_expr,numbers.Number):
            return 0
        if type(ind_expr)==Add:
            result = Add(*[self.abn_ind_der(A,i) for A in ind_expr.args])
            return result
        if type(ind_expr)==Mul:
            result=0
            for j in range(len(ind_expr.args)):
                result+=self.abn_ind_der(ind_expr.args[j],i)*Mul(*list(ind_expr.args[0:j]+ind_expr.args[j+1:len(ind_expr.args)]))
            return result
        if type(ind_expr)==Pow:
            return ind_expr.exp*ind_expr.base**(ind_expr.exp-1)*self.abn_ind_der(ind_expr.base,i)
        if type(ind_expr)==Indexed:
            base=ind_expr.base
            ind=list(ind_expr.indices)
            return base[ind+[i]]
        if type(ind_expr)==Symbol: return 0
        if type(ind_expr)==exp:
            return ind_expr*self.abn_ind_der(ind_expr.args[0],i)
    
    def are_deriv_normal(self,expr):
        """Returns True if all the directional derivatives of Indexed objects which appear in expr
        are in normal form; that is, they are expressed using only the fundamental directions, 3 and 4.
        INPUTS:
        * 'expr' - an algebraic expression in Indexed objects and symbols"""
        s=Indexed_obj_in_expr(expr)
        for A in s:
            if len(A.indices)>3:
                for i in A.indices[3:len(A.indices)]:
                    if i not in [3,4]: return False
        return True
        
    def normalize_der(self,expr):
        """Converts all derivatives to normal form using the bracket relations from self. Normal form means all derivatives
        are in the fundamental directions, directions 3 and 4.
        INPUTS:
        * 'expr' - an algebraic expression in Indexed objects and symbols
        
        Note: This only works for distributions with the symplectic symbol"""

        if self.are_deriv_normal(expr): return expr
        
        if isinstance(expr,numbers.Number): 
            return expr
        if type(expr)==Add:
            return Add(*[self.normalize_der(A) for A in expr.args])
        if type(expr)==Mul:
            return Mul(*[self.normalize_der(A) for A in expr.args])
        if type(expr)==Pow:
            return self.normalize_der(expr.base)**self.normalize_der(expr.exp)
        if type(expr)==Symbol: 
            return expr
        if type(expr)==exp:
            return exp(self.normalize_der(expr.args[0]))
        if type(expr)==Indexed:
            # Find the first abnormal direction k
            prefix=list(expr.indices[0:3])
            i=3
            while i<len(expr.indices) and expr.indices[i] in [3,4]:
                prefix.append(expr.indices[i])
                i+=1
            k=expr.indices[len(prefix)]
            postfix=list(expr.indices[len(prefix)+1:len(expr.indices)])

            # [Xi,Xj] = Xk in the Tanaka symbol
            if k in range(5,len(self.Tanaka_symbol.basis)-1):
                i,j=(3,k-1)
            if k==len(self.Tanaka_symbol.basis)-1:
                i,j=(5,len(self.Tanaka_symbol.basis)-3)

            no_post_r=K[prefix+[j,i]]-K[prefix+[i,j]] # Highest order terms, no postfix

            #Kurvs=(Xk-[Xi,Xj]).vec
            Xi=Vector_Field([0]*(i)+[1]+[0]*(len(self.Tanaka_symbol.basis)-i-1),self)
            Xj=Vector_Field([0]*(j)+[1]+[0]*(len(self.Tanaka_symbol.basis)-j-1),self)
            Xk=Vector_Field([0]*(k)+[1]+[0]*(len(self.Tanaka_symbol.basis)-k-1),self)
            Neg_Kurvs=(Xk-self.bracket(Xi,Xj)).vec
            for q in range(len(Neg_Kurvs)):
                no_post_r+=Neg_Kurvs[q]*K[prefix+[q]]

            r=self.normalize_der(no_post_r)
            for a in postfix:
                r=self.normalize_der(self.abn_ind_der(r,a))
            # Before substituting with Jacobi, I should make sure the values of Jacobi 
            # have only normal derivatives
            return r

In [87]:
class Vector_Field(object):
    def __init__(self,v_rep,parent):
        """INPUTS:
        * 'v_rep' - a list or vector representing a distribution in the frame of parent (positive and negative)
        * 'parent' - a Distr_of_constant_symbol object"""
        self.vec=Matrix(v_rep)
        if shape(self.vec)[1]!=1:
            self.vec=self.vec.transpose()
        self.parent=parent

    def bracket(self,other):
        return self.parent.bracket(self,other)
    
    def __str__(self):
        return str(self.vec)

    def __repr__(self):
        return self.vec.__repr__()
    
    def __add__(self,other):
        return Vector_Field(self.vec+other.vec,self.parent)
    
    def __neg__(self):
        return Vector_Field(-self.vec,self.parent)
    
    def __sub__(self,other):
        return self+(-other)